# Donkey Kong NES — CNN + DQN Agent
**UTS AI in Robotics — HD Submission**

Pipeline: `Game Screen → CNN (Person 2) → DQN Agent (Person 3) → Actions → PyBullet Arm (Person 1) → Evaluation (Person 4)`

### Before running
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your ROM when Cell 2 prompts you

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install stable-retro opencv-python pybullet matplotlib tqdm -q
# PyTorch with CUDA is pre-installed on Colab GPU runtimes
import torch
print(f'PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')

In [ ]:
# ── Cell 2: Upload ROM and import into stable-retro ───────────────────────────
from google.colab import files
import subprocess, sys, hashlib, os

print('Upload your ROM file: "Donkey Kong (World) (Rev 1).nes"')
uploaded = files.upload()   # pick the .nes file

rom_path = list(uploaded.keys())[0]
print(f'Uploaded: {rom_path}')

# Try standard retro import
result = subprocess.run([sys.executable, '-m', 'retro.import', rom_path],
                        capture_output=True, text=True)
print(result.stdout or result.stderr)

# Confirm game name
import retro
dk_games = [g for g in retro.data.list_games() if 'Donkey' in g or 'donkey' in g]
print('Detected Donkey Kong games:', dk_games)
GAME_NAME = dk_games[0] if dk_games else 'DonkeyKong-Nes'
print(f'Using game: {GAME_NAME}')

In [ ]:
# ── Cell 3: Person 1 — Environment wrapper ────────────────────────────────────
import collections
import numpy as np
import cv2
import retro

# NES buttons: B, A, SELECT, START, UP, DOWN, LEFT, RIGHT
DISCRETE_ACTIONS = [
    [0,0,0,0,0,0,0,0],  # 0 NOOP
    [0,0,0,0,0,0,1,0],  # 1 LEFT
    [0,0,0,0,0,0,0,1],  # 2 RIGHT
    [0,0,0,0,1,0,0,0],  # 3 UP
    [0,0,0,0,0,1,0,0],  # 4 DOWN
    [1,0,0,0,0,0,0,0],  # 5 JUMP
    [1,0,0,0,0,0,1,0],  # 6 JUMP+LEFT
    [1,0,0,0,0,0,0,1],  # 7 JUMP+RIGHT
]
NUM_ACTIONS  = len(DISCRETE_ACTIONS)
FRAME_H = FRAME_W = 84
FRAME_STACK = 4
MAX_STEPS   = 4500

class DonkeyKongEnv:
    def __init__(self, render=False):
        self.env = retro.make(
            game=GAME_NAME,
            render_mode='human' if render else None,
        )
        self.observation_shape = (FRAME_STACK, FRAME_H, FRAME_W)
        self._frames = collections.deque(maxlen=FRAME_STACK)
        self._reset_state()

    def _reset_state(self):
        self._prev_lives  = 3
        self._prev_mario_x = 0
        self._prev_mario_y = 0
        self._step_count  = 0

    def reset(self):
        obs = self.env.reset()
        if isinstance(obs, tuple): obs = obs[0]
        self._reset_state()
        frame = self._preprocess(obs)
        for _ in range(FRAME_STACK):
            self._frames.append(frame)
        return np.array(self._frames, dtype=np.float32)

    def step(self, action_idx):
        buttons = DISCRETE_ACTIONS[action_idx]
        result = self.env.step(buttons)
        obs, _, terminated, truncated = result[0], result[1], result[2], result[3]
        info = result[4] if len(result) > 4 else {}
        self._step_count += 1
        self._frames.append(self._preprocess(obs))
        reward = self._reward(info)
        done   = terminated or truncated or self._step_count >= MAX_STEPS
        self._update_prev(info)
        return np.array(self._frames, dtype=np.float32), reward, done, info

    def _preprocess(self, obs):
        gray    = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        resized = cv2.resize(gray, (FRAME_W, FRAME_H), interpolation=cv2.INTER_AREA)
        return resized.astype(np.float32) / 255.0

    def _reward(self, info):
        r = 0.0
        lives = info.get('lives', self._prev_lives)
        if lives < self._prev_lives:
            r -= 10.0
        mario_y = info.get('mario_y', self._prev_mario_y)
        dy = self._prev_mario_y - mario_y
        if dy > 0: r += dy * 0.1
        mario_x = info.get('mario_x', self._prev_mario_x)
        dx = mario_x - self._prev_mario_x
        if dx > 0: r += dx * 0.05
        r += 0.01
        return r

    def _update_prev(self, info):
        self._prev_lives   = info.get('lives',   self._prev_lives)
        self._prev_mario_x = info.get('mario_x', self._prev_mario_x)
        self._prev_mario_y = info.get('mario_y', self._prev_mario_y)

    def close(self):
        self.env.close()

# Quick smoke test
env = DonkeyKongEnv()
s = env.reset()
print(f'Env OK — obs shape: {s.shape}, actions: {NUM_ACTIONS}')
env.close()

In [ ]:
# ── Cell 4: Person 1 — PyBullet robot arm ────────────────────────────────────
import pybullet as p
import pybullet_data

_ACTION_JOINTS = {
    0: ( 0.00,  0.00),  # NOOP
    1: (-0.70,  0.00),  # LEFT
    2: ( 0.70,  0.00),  # RIGHT
    3: ( 0.00,  0.70),  # UP
    4: ( 0.00, -0.30),  # DOWN
    5: ( 0.00,  1.10),  # JUMP
    6: (-0.70,  1.10),  # JUMP+LEFT
    7: ( 0.70,  1.10),  # JUMP+RIGHT
}

class RobotArm:
    """2-DOF arm in PyBullet DIRECT mode (headless for Colab)."""
    def __init__(self):
        self.client = p.connect(p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath(), physicsClientId=self.client)
        p.setGravity(0, 0, -9.81, physicsClientId=self.client)
        p.loadURDF('plane.urdf', physicsClientId=self.client)
        self.arm = self._build_arm()

    def step(self, action_idx):
        base, shoulder = _ACTION_JOINTS.get(action_idx, (0.0, 0.0))
        p.setJointMotorControl2(self.arm, 0, p.POSITION_CONTROL,
                                targetPosition=base, force=500, physicsClientId=self.client)
        p.setJointMotorControl2(self.arm, 1, p.POSITION_CONTROL,
                                targetPosition=shoulder, force=500, physicsClientId=self.client)
        for _ in range(8):
            p.stepSimulation(physicsClientId=self.client)

    def close(self):
        p.disconnect(self.client)

    def _build_arm(self):
        bc = p.createCollisionShape(p.GEOM_CYLINDER, radius=0.10, height=0.20, physicsClientId=self.client)
        bv = p.createVisualShape(p.GEOM_CYLINDER, radius=0.10, length=0.20, rgbaColor=[.5,.5,.5,1], physicsClientId=self.client)
        l1c = p.createCollisionShape(p.GEOM_BOX, halfExtents=[.05,.05,.30], physicsClientId=self.client)
        l1v = p.createVisualShape(p.GEOM_BOX, halfExtents=[.05,.05,.30], rgbaColor=[.2,.6,.8,1], physicsClientId=self.client)
        l2c = p.createCollisionShape(p.GEOM_BOX, halfExtents=[.04,.04,.25], physicsClientId=self.client)
        l2v = p.createVisualShape(p.GEOM_BOX, halfExtents=[.04,.04,.25], rgbaColor=[.8,.4,.2,1], physicsClientId=self.client)
        arm = p.createMultiBody(
            baseMass=0, baseCollisionShapeIndex=bc, baseVisualShapeIndex=bv,
            basePosition=[0,0,.10],
            linkMasses=[1.0, 0.5],
            linkCollisionShapeIndices=[l1c, l2c], linkVisualShapeIndices=[l1v, l2v],
            linkPositions=[[0,0,.50],[0,0,.55]], linkOrientations=[[0,0,0,1],[0,0,0,1]],
            linkInertialFramePositions=[[0,0,0],[0,0,0]],
            linkInertialFrameOrientations=[[0,0,0,1],[0,0,0,1]],
            linkParentIndices=[0,1],
            linkJointTypes=[p.JOINT_REVOLUTE, p.JOINT_REVOLUTE],
            linkJointAxis=[[0,0,1],[0,1,0]],
            physicsClientId=self.client,
        )
        for i in range(2):
            p.setJointMotorControl2(arm, i, p.POSITION_CONTROL,
                                    targetPosition=0, force=500, physicsClientId=self.client)
        return arm

arm_test = RobotArm()
arm_test.step(2)
arm_test.close()
print('PyBullet arm OK')

In [ ]:
# ── Cell 5: Person 2 — CNN feature extractor ─────────────────────────────────
import torch
import torch.nn as nn

class CNNFeatureExtractor(nn.Module):
    """
    DeepMind DQN CNN (Nature 2015).
    Input  : (batch, 4, 84, 84) stacked grayscale frames
    Output : (batch, 512) feature vector

    Conv output sizes (84x84 input):
      Conv1 kernel=8 stride=4 → 20x20
      Conv2 kernel=4 stride=2 →  9x9
      Conv3 kernel=3 stride=1 →  7x7
      Flatten: 64*7*7 = 3136
    """
    def __init__(self, in_channels=4):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),          nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),          nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512), nn.ReLU(),
        )

    def forward(self, x):
        x = self.conv(x)
        return self.fc(x.view(x.size(0), -1))

class DQNNetwork(nn.Module):
    """CNN + linear head → Q-values per action."""
    def __init__(self, num_actions, in_channels=4):
        super().__init__()
        self.features = CNNFeatureExtractor(in_channels)
        self.head     = nn.Linear(512, num_actions)

    def forward(self, x):
        return self.head(self.features(x))

# Smoke test
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = DQNNetwork(NUM_ACTIONS).to(device)
dummy = torch.zeros(1, 4, 84, 84, device=device)
out = net(dummy)
print(f'CNN OK — input (1,4,84,84) → output {tuple(out.shape)}  [device: {device}]')

In [ ]:
# ── Cell 6: Person 3 — Replay buffer + DQN agent ─────────────────────────────
import random
import torch.optim as optim

class ReplayBuffer:
    def __init__(self, capacity, obs_shape):
        self.capacity = capacity
        self.pos = self.size = 0
        self.states      = np.zeros((capacity, *obs_shape), dtype=np.float32)
        self.next_states = np.zeros((capacity, *obs_shape), dtype=np.float32)
        self.actions     = np.zeros(capacity, dtype=np.int64)
        self.rewards     = np.zeros(capacity, dtype=np.float32)
        self.dones       = np.zeros(capacity, dtype=np.float32)

    def push(self, s, a, r, ns, d):
        self.states[self.pos]      = s
        self.next_states[self.pos] = ns
        self.actions[self.pos]     = a
        self.rewards[self.pos]     = r
        self.dones[self.pos]       = float(d)
        self.pos  = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = np.random.choice(self.size, batch_size, replace=False)
        return (self.states[idx], self.actions[idx], self.rewards[idx],
                self.next_states[idx], self.dones[idx])

    def __len__(self): return self.size


class DQNAgent:
    def __init__(self, num_actions, obs_shape, device,
                 lr=1e-4, gamma=0.99, batch_size=32,
                 buffer_capacity=100_000,
                 eps_start=1.0, eps_end=0.01, eps_decay_steps=500_000,
                 target_update_freq=1_000, train_freq=4, min_buffer=10_000):
        self.num_actions = num_actions
        self.device      = device
        self.gamma       = gamma
        self.batch_size  = batch_size
        self.eps         = eps_start
        self.eps_end     = eps_end
        self.eps_decay   = (eps_start - eps_end) / eps_decay_steps
        self.target_update_freq = target_update_freq
        self.train_freq  = train_freq
        self.min_buffer  = min_buffer
        self.step_count  = 0

        self.online_net  = DQNNetwork(num_actions).to(device)
        self.target_net  = DQNNetwork(num_actions).to(device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimizer   = optim.Adam(self.online_net.parameters(), lr=lr)
        self.loss_fn     = nn.SmoothL1Loss()
        self.buffer      = ReplayBuffer(buffer_capacity, obs_shape)

    def select_action(self, state):
        if random.random() < self.eps:
            return random.randrange(self.num_actions)
        with torch.no_grad():
            s = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            return int(self.online_net(s).argmax(dim=1).item())

    def observe(self, s, a, r, ns, done):
        self.buffer.push(s, a, r, ns, done)
        self.step_count += 1
        self.eps = max(self.eps_end, self.eps - self.eps_decay)
        loss = None
        if len(self.buffer) >= self.min_buffer and self.step_count % self.train_freq == 0:
            loss = self._train_step()
        if self.step_count % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())
        return loss

    def _train_step(self):
        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)
        states      = torch.FloatTensor(states).to(self.device)
        actions     = torch.LongTensor(actions).to(self.device)
        rewards     = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones       = torch.FloatTensor(dones).to(self.device)
        q_vals   = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            max_next = self.target_net(next_states).max(dim=1)[0]
            targets  = rewards + self.gamma * max_next * (1 - dones)
        loss = self.loss_fn(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online_net.parameters(), 10.0)
        self.optimizer.step()
        return loss.item()

    def save(self, path):
        torch.save({'online': self.online_net.state_dict(),
                    'target': self.target_net.state_dict(),
                    'opt':    self.optimizer.state_dict(),
                    'steps':  self.step_count, 'eps': self.eps}, path)

    def load(self, path):
        c = torch.load(path, map_location=self.device)
        self.online_net.load_state_dict(c['online'])
        self.target_net.load_state_dict(c['target'])
        self.optimizer.load_state_dict(c['opt'])
        self.step_count = c['steps']; self.eps = c['eps']

print('ReplayBuffer and DQNAgent defined OK')

In [ ]:
# ── Cell 7: Person 4 — Metrics tracker ───────────────────────────────────────
import json

class MetricsTracker:
    def __init__(self):
        self.episode_rewards = []
        self.episode_steps   = []
        self.episode_deaths  = []
        self.episode_losses  = []
        self.episode_eps     = []

    def log(self, reward, steps, deaths, loss, eps):
        self.episode_rewards.append(reward)
        self.episode_steps.append(steps)
        self.episode_deaths.append(deaths)
        self.episode_losses.append(loss)
        self.episode_eps.append(eps)

    def smoothed(self, data, w=20):
        return [np.mean(data[max(0,i-w+1):i+1]) for i in range(len(data))]

    def success_rate(self, w=100):
        r = self.episode_steps[-w:]
        return sum(s > 2000 for s in r) / len(r) if r else 0.0

    def save(self, path='/content/metrics.json'):
        with open(path,'w') as f:
            json.dump({'rewards': self.episode_rewards, 'steps': self.episode_steps,
                       'deaths': self.episode_deaths,  'losses': self.episode_losses,
                       'eps':    self.episode_eps}, f)
        print(f'Metrics saved → {path}')

print('MetricsTracker defined OK')

In [ ]:
# ── Cell 8: Training loop ─────────────────────────────────────────────────────
import os
from tqdm.notebook import tqdm

# ── Hyperparameters ── tweak here ──
NUM_EPISODES  = 1000   # reduce to 200 for a quick smoke test
SAVE_FREQ     = 100
SAVE_DIR      = '/content/saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)
# ──────────────────────────────────

env     = DonkeyKongEnv(render=False)
arm     = RobotArm()
agent   = DQNAgent(NUM_ACTIONS, env.observation_shape, device)
metrics = MetricsTracker()

for episode in tqdm(range(1, NUM_EPISODES + 1), desc='Training'):
    state       = env.reset()
    ep_reward   = 0.0
    ep_losses   = []
    ep_steps    = 0
    ep_deaths   = 0
    prev_lives  = 3
    done        = False

    while not done:
        action = agent.select_action(state)
        arm.step(action)                             # mirror action on robot arm
        next_state, reward, done, info = env.step(action)

        lives = info.get('lives', prev_lives)
        if lives < prev_lives: ep_deaths += 1
        prev_lives = lives

        loss = agent.observe(state, action, reward, next_state, done)
        if loss is not None: ep_losses.append(loss)

        state      = next_state
        ep_reward += reward
        ep_steps  += 1

    avg_loss = float(np.mean(ep_losses)) if ep_losses else 0.0
    metrics.log(ep_reward, ep_steps, ep_deaths, avg_loss, agent.eps)

    if episode % SAVE_FREQ == 0:
        ckpt = f'{SAVE_DIR}/dqn_ep{episode}.pt'
        agent.save(ckpt)
        print(f'\nEp {episode:4d} | avg_r={np.mean(metrics.episode_rewards[-100:]):.1f} | eps={agent.eps:.3f} | saved {ckpt}')

env.close()
arm.close()
metrics.save()
print('Training complete!')

In [ ]:
# ── Cell 9: Person 4 — Evaluation plots ──────────────────────────────────────
import matplotlib.pyplot as plt

m = metrics   # or: load from metrics.json with json.load
n = len(m.episode_rewards)
eps_axis = list(range(1, n + 1))
W = 50

def rolling(data, w):
    return [np.mean(data[max(0,i-w+1):i+1]) for i in range(len(data))]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('Donkey Kong DQN — Training Metrics', fontsize=15, fontweight='bold')

# 1 Reward curve
ax = axes[0,0]
ax.plot(eps_axis, m.episode_rewards, alpha=0.3, color='steelblue', lw=0.8)
ax.plot(eps_axis, rolling(m.episode_rewards, 20), color='steelblue', lw=2, label='Smoothed (w=20)')
ax.set_title('Reward Curve'); ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward')
ax.legend(); ax.grid(True, alpha=0.3)

# 2 Success rate
ax = axes[0,1]
sr = [sum(s>2000 for s in m.episode_steps[max(0,i-W+1):i+1]) /
      len(m.episode_steps[max(0,i-W+1):i+1]) for i in range(n)]
ax.plot(eps_axis, sr, color='green', lw=2)
ax.set_title(f'Success Rate (survived >2000 steps, rolling w={W})')
ax.set_xlabel('Episode'); ax.set_ylabel('Rate'); ax.set_ylim(0,1); ax.grid(True, alpha=0.3)

# 3 Death rate
ax = axes[1,0]
ax.plot(eps_axis, m.episode_deaths, alpha=0.3, color='crimson', lw=0.8)
ax.plot(eps_axis, rolling(m.episode_deaths, W), color='crimson', lw=2, label=f'Rolling avg (w={W})')
ax.set_title('Collision / Death Rate'); ax.set_xlabel('Episode'); ax.set_ylabel('Deaths / Episode')
ax.legend(); ax.grid(True, alpha=0.3)

# 4 Steps to goal
ax = axes[1,1]
ax.plot(eps_axis, m.episode_steps, alpha=0.3, color='darkorange', lw=0.8)
ax.plot(eps_axis, rolling(m.episode_steps, 20), color='darkorange', lw=2, label='Smoothed (w=20)')
ax.set_title('Steps per Episode'); ax.set_xlabel('Episode'); ax.set_ylabel('Steps')
ax.legend(); ax.grid(True, alpha=0.3)

# 5 Loss curve
ax = axes[2,0]
lv = [(i+1, l) for i,l in enumerate(m.episode_losses) if l > 0]
if lv:
    lx, ly = zip(*lv)
    ax.plot(lx, ly, alpha=0.3, color='purple', lw=0.8)
    ax.plot(lx, rolling(list(ly), 20), color='purple', lw=2, label='Smoothed (w=20)')
ax.set_title('Loss Curve (Huber)'); ax.set_xlabel('Episode'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

# 6 Epsilon decay
ax = axes[2,1]
ax.plot(eps_axis, m.episode_eps, color='teal', lw=2)
ax.set_title('Epsilon (Exploration Rate)'); ax.set_xlabel('Episode')
ax.set_ylabel('Epsilon'); ax.set_ylim(0,1); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_metrics.png', dpi=150)
plt.show()

# Summary stats
last = slice(max(0, n-100), n)
print('\n========== Evaluation Summary ==========')
print(f'  Total episodes          : {n}')
print(f'  Avg reward  (last 100)  : {np.mean(m.episode_rewards[last]):.2f}')
print(f'  Success rate (last 100) : {m.success_rate(100):.1%}')
print(f'  Avg deaths  (last 100)  : {np.mean(m.episode_deaths[last]):.2f}')
print(f'  Avg steps   (last 100)  : {np.mean(m.episode_steps[last]):.1f}')
print('=========================================')

In [ ]:
# ── Cell 10: Download results ─────────────────────────────────────────────────
from google.colab import files

files.download('/content/metrics.json')
files.download('/content/training_metrics.png')

# Download latest checkpoint
import glob
checkpoints = sorted(glob.glob('/content/saved_models/*.pt'))
if checkpoints:
    files.download(checkpoints[-1])
    print(f'Downloaded: {checkpoints[-1]}')